In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start with sk-proj-; please check your are using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
 

In [ ]:
OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.6"
}
MODEL = "llama3.2"

In [ ]:
class Website:
    def __init__(self, url):
        """
        Create this website object from the given url using BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"

        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()

        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
web = Website("https://www.trailadventours.com/")
print(web.title)
print(web.text)

In [ ]:
system_prompt = "You are an assistant that analyzes the contents of a website" \
"and provides a short summary, ignoring text that might be navigation related. " \
"Respond in markdown."

In [ ]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a Philippines hiking website titled name {website.title}"

    user_prompt += "\nHer creativity of her works are as follow;\
        please provide a short summary of her works in markdown. \n"
    user_prompt += website.text
    return user_prompt

In [ ]:
print(user_prompt_for(web))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_for(web)},
]

In [ ]:
payload = {
    "model": MODEL,
    "messages": messages,
    "stream": False
}

In [ ]:
response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
print(response.json()["message"]["content"])

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(web)}
    ]

In [ ]:
messages_for(web)

In [ ]:
def summarize(url):
    website = Website(url)
    response = requests.post(
        OLLAMA_API,
        json={
            "model": MODEL,
            "messages": messages_for(website),
            "stream": False
        },
        headers=HEADERS
    )
    return response.json()["message"]["content"]

In [ ]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://www.trailadventours.com/")

In [ ]:
# from openai import OpenAI
# openai = OpenAI()

In [ ]:
# requests.get("http://localhost:11434").content

In [ ]:
# !ollama pull llama3.2

In [ ]:
# OLLAMA_BASE_URL = "http://localhost:11434/v1"

# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
# response = ollama.chat.completions.create(model="llama3.2", messages=[
#     {"role": "user", "content": "Tell me a fun fact"}
# ])

# response.choices[0].message.content